In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Consistency Evaluation — Binary Checklist

This notebook evaluates whether the research project at `/net/scratch2/smallyan/function_vectors_eval` meets its stated goal based on five consistency criteria:

1. **CS1**: Conclusion vs Original Results
2. **CS2**: Implementation Follows the Plan
3. **CS3**: Effect Size
4. **CS4**: Justification of Steps and Intermediate Conclusions
5. **CS5**: Statistical Significance Reporting

In [2]:
# Check CUDA availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU device: NVIDIA A40


In [3]:
# Explore the repository structure
repo_path = '/net/scratch2/smallyan/function_vectors_eval'
for root, dirs, files in os.walk(repo_path):
    # Skip hidden directories and common non-essential directories
    dirs[:] = [d for d in dirs if not d.startswith('.') and d not in ['__pycache__', '.git']]
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

function_vectors_eval/
  .gitignore
  fv_overview.png
  documentation.pdf
  plan.md
  CodeWalkthrough.md
  fv_environment.yml
  no_exe_evaluation/
    code_critic_evaluation.ipynb
    code_critic_summary.json
    replications/
      self_replication_evaluation.json
      no_exe_evaluation_replication.md
  doc_only_evaluation/
    self_matching.ipynb
    generalization_eval_summary.json
    generalization_eval.ipynb
    consistency_evaluation.json
  src/
    portability_eval.py
    test_numheads.py
    compute_indirect_effect.py
    vocab_reconstruction.py
    __init__.py
    compute_avg_hidden_state.py
    natural_text_eval.py
    evaluate_function_vector.py
    compute_average_activations.py
    utils/
      eval_utils.py
      prompt_utils.py
      intervention_utils.py
      extract_utils.py
      __init__.py
      model_utils.py
    eval_scripts/
      eval_fv.sh
      eval_numheads.sh
      eval_template_portability.sh
      eval_avg_hs.sh
      template.sh
      fv_eval_sweep.py


## Repository Structure

The repository contains:
- **plan.md**: Project plan
- **documentation.pdf**: Project documentation
- **CodeWalkthrough.md**: Code walkthrough
- **notebooks/fv_demo.ipynb**: Main demonstration notebook
- **src/**: Source code implementation
- **dataset_files/**: Dataset files for evaluation

In [4]:
# Read the plan file
plan_path = os.path.join(repo_path, 'plan.md')
with open(plan_path, 'r') as f:
    plan_content = f.read()
print(plan_content)

# Plan
## Objective
To investigate whether autoregressive transformer language models develop compact vector representations of input-output functions (called function vectors) within their hidden states during in-context learning, and to characterize these representations across diverse tasks and models.

## Hypothesis
1. A small number of attention heads transport a compact representation of the demonstrated task (function vector) that is robust to changes in context and can trigger task execution in zero-shot and natural text settings.
2. Function vectors contain information encoding the output space of the function, but this information alone is not sufficient to reconstruct a working function vector.
3. Function vectors can be composed through vector algebra to create vectors that trigger new complex tasks combining constituent tasks.

## Methodology
1. Apply causal mediation analysis to identify attention heads with highest average indirect effect (AIE) across diverse ICL tasks. 

In [5]:
# Read the CodeWalkthrough file
code_walkthrough_path = os.path.join(repo_path, 'CodeWalkthrough.md')
with open(code_walkthrough_path, 'r') as f:
    code_walkthrough_content = f.read()
print(code_walkthrough_content)

# Function Vectors in Large Language Models
### [Project Website](https://functions.baulab.info) | [Arxiv Preprint](https://arxiv.org/abs/2310.15213) | [OpenReview](https://openreview.net/forum?id=AwyxtyMwaG)

This repository contains data and code for the paper: [Function Vectors in Large Language Models](https://arxiv.org/pdf/2310.15213).

<p align="left">
<img src="https://functions.baulab.info/images/Paper/fv-demonstrations.png" style="width:100%;"/>
</p> 

## Setup

We recommend using conda as a package manager. 
The environment used for this project can be found in the `fv_environment.yml` file.
To install, you can run: 
```
conda env create -f fv_environment.yml
conda activate fv
```

## Demo Notebook
Checkout `notebooks/fv_demo.ipynb` for a jupyter notebook with a demo of how to create a function vector and use it in different contexts.

## Data
The datasets used in our project can be found in the `dataset_files` folder.

## Code
Our main evaluation scripts are contained in the

In [6]:
# Extract text from documentation.pdf
import subprocess
doc_path = os.path.join(repo_path, 'documentation.pdf')

# Use pdftotext to extract text
result = subprocess.run(['pdftotext', doc_path, '-'], capture_output=True, text=True)
doc_text = result.stdout
print(doc_text)

FileNotFoundError: [Errno 2] No such file or directory: 'pdftotext'

In [7]:
# Try using PyPDF2 or pypdf to extract text from the PDF
try:
    import pypdf
    with open(doc_path, 'rb') as f:
        reader = pypdf.PdfReader(f)
        doc_text = ""
        for page in reader.pages:
            doc_text += page.extract_text() + "\n"
    print(doc_text)
except ImportError:
    try:
        import PyPDF2
        with open(doc_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            doc_text = ""
            for page in reader.pages:
                doc_text += page.extract_text() + "\n"
        print(doc_text)
    except ImportError:
        print("Neither pypdf nor PyPDF2 is available")

Published as a conference paper at ICLR 2024
FUNCTION VECTORS IN LARGE LANGUAGE MODELS
Eric Todd∗, Millicent L. Li, Arnab Sen Sharma, Aaron Mueller,
Byron C. Wallace, and David Bau
Khoury College of Computer Sciences, Northeastern University
ABSTRACT
We report the presence of a simple neural mechanism that represents an input-
output function as a vector within autoregressive transformer language models
(LMs). Using causal mediation analysis on a diverse range of in-context-learning
(ICL) tasks, we find that a small number attention heads transport a compact
representation of the demonstrated task, which we call a function vector (FV). FVs
are robust to changes in context, i.e., they trigger execution of the task on inputs
such as zero-shot and natural text settings that do not resemble the ICL contexts
from which they are collected. We test FVs across a range of tasks, models, and
layers and find strong causal effects across settings in middle layers. We investigate
the internal struc

In [8]:
# Let's extract relevant sections from the documentation PDF more carefully
import pypdf

doc_path = os.path.join(repo_path, 'documentation.pdf')

# Extract all text
with open(doc_path, 'rb') as f:
    reader = pypdf.PdfReader(f)
    pages_text = []
    for i, page in enumerate(reader.pages):
        pages_text.append((i+1, page.extract_text()))
    
# Print page count
print(f"Total pages: {len(pages_text)}")

# Let's print the structure - first few lines of each page to understand what's in each
for page_num, text in pages_text:
    lines = text.split('\n')[:5]
    print(f"\n--- Page {page_num} ---")
    for line in lines:
        print(line[:100])

ModuleNotFoundError: No module named 'pypdf'